# 05 - Agent Improvement With Prompts and Higher-Level Tools

This notebook documents the first reliability iteration for the Ollama-backed Notely AI PM Analytics Agent.

The goal is not only to make the answer better. The goal is to show the engineering loop: observe a weak agent answer, diagnose why it failed, add constraints and better tools, then rerun the same question.

## Baseline Answer From The First Ollama Run

Question:

> why did IOS activation rate change around mid May 2026?

Baseline answer:

> Based on the product context, it appears that there was a launch of the Action items feature around April 15th, and an incident with an iOS upload retry bug began around May 10th. These events may have contributed to changes in the IOS activation rate around mid-May 2026.

> To further investigate, I would recommend running a SQL query to analyze the impact of these events on the IOS activation rate. For example: SELECT ios_activation_rate FROM analytics WHERE calendar_date BETWEEN '2026-05-01' AND '2026-05-15'.

Tool trace: the model only called `get_product_context` for 2026-04-01 to 2026-05-15.

## Why The Baseline Answer Is Not Good Enough

The first run is technically successful because Ollama called a tool and retrieved relevant product context. But the analytical answer is incomplete.

Problems:

- It did not pull the actual activation rate before and after mid May.
- It did not check the adjacent funnel metric, upload completion rate.
- It suggested a fake SQL query against a fake `analytics` table and fake `ios_activation_rate` column.
- It gave a plausible story before grounding the answer in metric evidence.

This is a classic early-agent issue: the model can call tools, but it does not yet follow a reliable analysis workflow.

## What This Notebook Is Solving

This iteration improves reliability in two ways:

1. Prompt constraints: the system prompt now tells the model not to invent SQL, tables, columns, numbers, or metric definitions.
2. Higher-level tool: `diagnose_metric_change` packages the common PM workflow into one tool.

For a question like 'why did iOS activation rate change around mid May?', the agent should now call `diagnose_metric_change` with roughly:

```python
diagnose_metric_change(
    metric_name='activation_rate',
    change_date='2026-05-10',
    segment='iOS',
    group_by='platform',
    window_days=7,
)
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## Test The New Diagnostic Tool Directly

Before asking the LLM to use the tool, test the deterministic Python tool by itself. This separates tool correctness from model behavior.

In [ ]:
from agent.tools import diagnose_metric_change

diagnosis = diagnose_metric_change(
    metric_name='activation_rate',
    change_date='2026-05-10',
    segment='iOS',
    group_by='platform',
    window_days=7,
)
diagnosis['target_metric']

In [ ]:
diagnosis['adjacent_metrics']

In [ ]:
diagnosis['product_context']['rows']

## Rerun The Agent

After the prompt and tool changes, rerun the same question from Terminal:

```bash
python3 llm_agent_demo.py --show-trace "why did IOS activation rate change around mid May 2026?"
```

A better trace should include `diagnose_metric_change`. A better answer should include before/after activation rate values, upload completion as a funnel driver, and the iOS upload retry bug as product context.

## Paste The New Output Here

Use this section to compare the next Ollama output with the baseline.

Checklist:

- Did it call `diagnose_metric_change`?
- Did it include before vs after activation rate values?
- Did it mention upload completion rate as adjacent funnel evidence?
- Did it avoid fake SQL, fake tables, and fake columns?
- Did the final explanation sound like a PM analytics insight?